# Correr Test 2 ($S_n$, función característica empírica)

Este notebook permite ejecutar el estudio Monte Carlo del Test 2 con parámetros configurables. Las salidas se guardan en `results/data/`.

**Nota:** $S_n$ es ~5-10× más lento por test que $T_n$ porque cada evaluación requiere cuadratura sobre un grid de $K$ puntos. El estimador `argmin` consume ~85% del CPU total.

- **Quick** (~7-10 min): B=199, R=60, 2 pesos, n=(20,40,80)
- **Full** (~2.5 h): B=500, R=200, 3 pesos, n=(20,40,80,160)

Ver `documentacion/documentacion_test2.pdf` para más detalle.

In [ ]:
# --- Setup ---
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.distributions import default_h0_specs, default_ha_specs
from src.simulation_sn import SimConfigSn, summarize_sn
from src.characteristic_fn import default_weights

print(f'Working dir: {Path.cwd()}')
print(f'CPU cores: {os.cpu_count()}')
print(f'Pesos disponibles por defecto: {list(default_weights().keys())}')

## Parámetros

Edita esta celda para cambiar la configuración.

In [ ]:
# --- Parámetros ---
quick = True              # True: rápido (~7-10 min). False: full (~2.5 h).
use_parallel = True       # True: ProcessPoolExecutor. False: secuencial.

if quick:
    config = SimConfigSn(
        sample_sizes=(20, 40, 80),
        estimators=('argmin', 'median', 'trimmed'),
        qs=(1, 2),
        weight_names=('gauss_1.0', 'laplace_1.0'),
        B=199, R=60, alpha=0.05, seed=2026, n_t_grid=201,
    )
    suffix = '_quick'
else:
    config = SimConfigSn(
        sample_sizes=(20, 40, 80, 160),
        estimators=('argmin', 'median', 'trimmed'),
        qs=(1, 2),
        weight_names=('gauss_1.0', 'gauss_0.5', 'laplace_1.0'),
        B=500, R=200, alpha=0.05, seed=2026, n_t_grid=301,
    )
    suffix = ''

specs = default_h0_specs() + default_ha_specs()
print(f'Distribuciones: {[s.name for s in specs]}')
print(f'Tamaños n: {config.sample_sizes}')
print(f'Estimadores: {config.estimators}')
print(f'q: {config.qs}, pesos: {config.weight_names}')
print(f'B={config.B}, R={config.R}, K (grid t)={config.n_t_grid}')
total = (len(specs) * len(config.sample_sizes) * len(config.estimators)
         * len(config.qs) * len(config.weight_names) * config.R)
print(f'Tareas totales: {total:,}')

## Ejecutar la simulación

In [ ]:
# --- Run ---
if use_parallel:
    from src.simulation_sn_parallel import run_simulation_sn_parallel
    n_workers = max(1, (os.cpu_count() or 2) - 1)
    df = run_simulation_sn_parallel(specs, config, n_workers=n_workers)
else:
    from src.simulation_sn import run_simulation_sn
    df = run_simulation_sn(specs, config)

summary = summarize_sn(df, alpha=config.alpha)
print(f'\nFilas raw: {len(df):,}  |  Filas summary: {len(summary)}')

## Guardar resultados

In [ ]:
data_dir = ROOT / 'results' / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
raw_csv = data_dir / f'sn_simulation_raw{suffix}.csv'
sum_csv = data_dir / f'sn_simulation_summary{suffix}.csv'
df.to_csv(raw_csv, index=False)
summary.to_csv(sum_csv, index=False)
print(f'Guardado:\n  {raw_csv}\n  {sum_csv}')

## Resumen rápido

In [ ]:
import pandas as pd

print('Tasas de rechazo (argmin, q=2; columnas = peso):')
sub = summary[(summary['estimator']=='argmin') & (summary['q']==2)]
if not sub.empty:
    display(sub.pivot_table(index=['dist','n'], columns='weight', values='reject_rate').round(3))
else:
    print('(sin datos para argmin q=2)')

In [ ]:
print('Tiempo medio por test (s), por estimador y q:')
summary.pivot_table(index=['estimator','n'], columns='q', values='mean_time_s').round(3)

Para generar gráficas usar el notebook `04_plots_test2.ipynb`.